# DataCite citations

Run this either everytime new DataCite datasets are pulled or to update citations of older datasets.

The DataCite metadata comes with citations to datasets, if any (DOI only).
Here we process them to get their publication year and generate the citation file with weights

In [51]:
%load_ext autoreload
%autoreload 2
from sindex.sources.datacite.jobs import (
    batch_find_citations_from_dc_serial,
    batch_slim_datacite_chunked,
    extract_unique_dois_from_citation_blocks,
    harvest_datacite_datasets_for_date_range_to_ndjson,
    lookup_dates_in_oa_snapshot,
)
from sindex.sources.datacite.utils import (
    get_citation_blocks_from_ndjson,
    get_new_citation_blocks,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Processing citations of newly pulled DataCite datasets 

### Extract citation blocks from slim metadata files for easier processing

In [45]:
datacite_slim_dir = r"D:\may-2026-data\records\slim-records\datacite-slim-records"
citation_blocks_file = r"D:\may-2026-data\citations\datacite\citation_blocks.ndjson"

In [46]:
get_citation_blocks_from_ndjson(datacite_slim_dir, citation_blocks_file)

[2026-06-04 10:22:44.948983] Processing 213 files...
[213/213] slim-99.ndjson â€” 96,968 saved so farr
[2026-06-04 10:27:45.568010] Complete! Total Records Saved: 96,968


### Get a list of unique citing DOIs for efficiently querying their publication year

In [8]:
citations_parquet_file = (
    r"D:\may-2026-data\citations\datacite\datacite_unique_citation_doi.parquet"
)

In [10]:
extract_unique_dois_from_citation_blocks(citation_blocks_file, citations_parquet_file)

Scanning citation_blocks.ndjson...
Scanned 96,968 records. Unique DOIs: 50,876
Exporting 50,876 unique DOIs to D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet...
[SUCCESS] Saved to D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet


### Get publication dates from OpenAlex snapshot if available

In [15]:
# Paths
citations_parquet_file_with_pubdates = (
    r"D:\may-2026-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"
)
oa_db_path = r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb"

In [14]:
lookup_dates_in_oa_snapshot(
    oa_db_path, citations_parquet_file, citations_parquet_file_with_pubdates
)

Joining D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet with OpenAlex database...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    [SUCCESS] Found 19,376 matches.
    [INFO] Results saved to: D:\may-2026-data\citations\datacite\datacite_citation_dois_with_pubyears.parquet
    [INFO] Time taken: 30.27s


### Create citations file

In [17]:
out_ndjson = r"D:\may-2026-data\citations\datacite\dc_citations.ndjson"

In [20]:
batch_find_citations_from_dc_serial(
    citation_blocks_file, out_ndjson, citations_parquet_file_with_pubdates
)

[2026-06-01 18:55:16.786371] Starting SERIAL processing...
[*] Loading cache from datacite_citation_dois_with_pubdates.parquet...
    -> Loaded 19,376 dates into memory.
[*] Counting exact lines in input file...
    -> Total workload: 96,968 lines.
[*] Processing lines...
Progress: 99.00% | Found: 112,722

[DONE] Finished in 68190.78s.
       Total Lines Processed: 96,968
       Total Citations Found: 113,883


## Process citations of previously saved DataCite datasets

### Get records of datasets previously harvested that have been updated in DataCite since then

In [36]:
start_date_str = "2011-03-08"
end_date_str = "2025-09-30"
updated_after = "2025-10-01"
datacite_updated_records_folder = r"D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-raw-records"

In [33]:
harvest_datacite_datasets_for_date_range_to_ndjson(
    start_date_str=start_date_str,
    end_date_str=end_date_str,
    updated_after=updated_after,
    citations_only=True,
    save_folder=datacite_updated_records_folder,
)

Fetching records 2025-09-24 â†’ 2025-09-30 (window_days=7, page_size=1000, detail=True)
  Saved 2247 records â†’ D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-2025-09-24-2025-09-30.ndjson
Fetching records 2025-09-17 â†’ 2025-09-23 (window_days=7, page_size=1000, detail=True)
  Saved 2163 records â†’ D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-2025-09-17-2025-09-23.ndjson
Fetching records 2025-09-10 â†’ 2025-09-16 (window_days=7, page_size=1000, detail=True)
  Saved 1620 records â†’ D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-2025-09-10-2025-09-16.ndjson
Fetching records 2025-09-03 â†’ 2025-09-09 (window_days=7, page_size=1000, detail=True)
  Saved 1309 records â†’ D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-2025-09-03-2025-09-09.ndjson
Fetching records 2025-08-27 â†’ 2025-09-02 (window_days=7, page_size=1000, detail=True)
  Saved 4885 records â†’ D:\

999189

### Create slim records

In [41]:
datacite_updated_slim_folder = r"D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-slim-records"

In [40]:
summary = batch_slim_datacite_chunked(
    src_folder=datacite_updated_records_folder, dst_folder=datacite_updated_slim_folder
)

Found 771 input files.
Starting processing with 32 cores. Batch size: 100,000...
Processed: 999,189 lines | Batches: 10 | Bad: 0
 
DONE in 67.59s
Total Lines Read: 999,189
Total Lines Kept: 999,189
Processing Rate:  14,782 records/sec
Output Files:     10 files written to D:\may-2026-data\citations\datacite-previous\datacite-updated-records\datacite-slim-records
 


### Get citation blocks from the new records

In [42]:
citation_blocks_updated = (
    r"D:\may-2026-data\citations\datacite-previous\citation_blocks_updated.ndjson"
)

In [43]:
get_citation_blocks_from_ndjson(
    ndjson_folder=datacite_updated_slim_folder,
    output_file_path=citation_blocks_updated,
)

[2026-06-04 10:21:31.096754] Processing 10 files...
[10/10] slim-9.ndjson â€” 999,189 saved so far
[2026-06-04 10:21:56.791551] Complete! Total Records Saved: 999,189


### Compare with previous citation blocks to only retain new ones

In [52]:
citation_blocks_existing = [
    r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
]
final_citation_blocks = (
    r"D:\may-2026-data\citations\datacite-previous\citation_blocks.ndjson"
)
dataset_db_path = r"D:/combined-data/records/slim-records/datacite-slim-records.duckdb"
datasets_added_before = "2026-01-02"

In [54]:
get_new_citation_blocks(
    existing_citations_paths=citation_blocks_existing,
    update_citations_path=citation_blocks_updated,
    output_file_path=final_citation_blocks,
    dataset_db_path=dataset_db_path,
    datasets_added_before=datasets_added_before,
)

[2026-06-04 10:34:28.197437] Loading known dataset IDs from D:/combined-data/records/slim-records/datacite-slim-records.duckdb...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-06-04 10:35:07.509651] Loaded 49,009,522 known dataset IDs added before 2026-01-02.
[2026-06-04 10:35:07.509651] Loading existing citation pairs from 1 file(s)...
  citation_blocks.ndjson: 1,565,793 records loaded.
[2026-06-04 10:36:36.592216] 1,565,793 unique datasets across all existing files. Comparing updates...
[2026-06-04 10:36:46.060980] Done. 27,451 datasets with new citations, 821,118 new citation pairs total, 1,795 skipped (not in known datasets).


### Get a list of unique citing DOIs for efficiently querying their publication year

In [57]:
citations_parquet_file = (
    r"D:\may-2026-data\citations\datacite-previous\datacite_unique_citation_doi.parquet"
)

In [58]:
extract_unique_dois_from_citation_blocks(final_citation_blocks, citations_parquet_file)

Scanning citation_blocks.ndjson...
Scanned 27,451 records. Unique DOIs: 50,862
Exporting 50,862 unique DOIs to D:\may-2026-data\citations\datacite-previous\datacite_unique_citation_doi.parquet...
[SUCCESS] Saved to D:\may-2026-data\citations\datacite-previous\datacite_unique_citation_doi.parquet


### Get publication dates from OpenAlex snapshot if available

In [59]:
# Paths
citations_parquet_file_with_pubdates = (
    r"D:\may-2026-data\citations\datacite-previous\datacite_citation_dois_with_pubdates.parquet"
)
oa_db_path = r"D:\combined-data\external\openalex-snapshot\oa_snapshot.duckdb"

In [60]:
lookup_dates_in_oa_snapshot(
    oa_db_path, citations_parquet_file, citations_parquet_file_with_pubdates
)

Joining D:\may-2026-data\citations\datacite-previous\datacite_unique_citation_doi.parquet with OpenAlex database...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    [SUCCESS] Found 49,480 matches.
    [INFO] Results saved to: D:\may-2026-data\citations\datacite-previous\datacite_citation_dois_with_pubdates.parquet
    [INFO] Time taken: 176.93s


### Create citations file

In [63]:
out_ndjson = r"D:\may-2026-data\citations\datacite-previous\dc_citations.ndjson"

In [64]:
batch_find_citations_from_dc_serial(
    final_citation_blocks, out_ndjson, citations_parquet_file_with_pubdates
)

[2026-06-04 11:18:15.272815] Starting SERIAL processing...
[*] Loading cache from datacite_citation_dois_with_pubdates.parquet...
    -> Loaded 49,480 dates into memory.
[*] Counting exact lines in input file...
    -> Total workload: 27,451 lines.
[*] Processing lines...
Progress: 98.36% of datasets | Found: 820,562 citations

[DONE] Finished in 6536.27s.
       Total Lines Processed: 27,451
       Total Citations Found: 821,118
